In [1]:
import pandas as pd

from analytics.db import query
from analytics.paths import RAW_OLIST

# Day 6 — pandas, Past the Basics

## Objective

Use appropriate data types, validated merges, aggregation, transformation,
reshaping, and method chaining to analyze the Olist dataset.

The main question is: Which product categories grew the most from October
to November 2017?

## Law of the Day

A clear error is better than a wrong number.

In [2]:
query("""
      --w06
    SELECT
        seller_city,
        seller_state,
        COUNT(*) AS sellers_count
    FROM sellers
    GROUP BY seller_city, seller_state
    ORDER BY sellers_count DESC, seller_city ASC, seller_state ASC
    LIMIT 10;
""")

,seller_city,seller_state,sellers_count
0,sao paulo,SP,694
1,curitiba,PR,124
2,rio de janeiro,RJ,93
3,belo horizonte,MG,66
4,ribeirao preto,SP,52
5,guarulhos,SP,50
6,ibitinga,SP,49
7,santo andre,SP,45
8,campinas,SP,41
9,maringa,PR,40


In [3]:
orders_csv = pd.read_csv(RAW_OLIST / "olist_orders_dataset.csv")
orders = query("SELECT * FROM orders")

orders_csv.shape, orders.shape

((99441, 8), (99441, 8))

Both DataFrames contain 99,441 rows and 8 columns.
Matching dimensions alone do not prove that all values are identical.

In [4]:
def megabytes(df):
    return round(df.memory_usage(deep=True).sum() / 1_000_000, 1)

megabytes(orders_csv), megabytes(orders)

(np.float64(55.5), np.float64(25.9))

The megabytes function estimates DataFrame memory usage in decimal MB,
including the index and deep inspection of text values.
It does not measure CSV file size or PostgreSQL storage size.

In [5]:
pd.DataFrame({
    "from_csv": orders_csv.dtypes,
    "from_db": orders.dtypes,
})

,from_csv,from_db
order_id,str,str
customer_id,str,str
order_status,str,str
order_purchase_timestamp,str,datetime64[us]
order_approved_at,str,datetime64[us]
order_delivered_carrier_date,str,datetime64[us]
order_delivered_customer_date,str,datetime64[us]
order_estimated_delivery_date,str,datetime64[us]


In [6]:
smaller = orders.astype({"order_status": "category"})

megabytes(smaller)

np.float64(20.2)

I created smaller by converting order_status to category.
The original orders DataFrame was kept unchanged.
Categorical storage can reduce memory usage for repeated status values.

In [7]:
orders_csv.memory_usage(deep=True)

Index                                132
order_id                         8054721
customer_id                      8054721
order_status                     5765932
order_purchase_timestamp         6761988
order_approved_at                6756228
order_delivered_carrier_date     6697800
order_delivered_customer_date    6655248
order_estimated_delivery_date    6761988
dtype: int64

In [8]:
orders.memory_usage(deep=True)

Index                                132
order_id                         8054721
customer_id                      8054721
order_status                     5765932
order_purchase_timestamp          795528
order_approved_at                 795528
order_delivered_carrier_date      795528
order_delivered_customer_date     795528
order_estimated_delivery_date     795528
dtype: int64

In [9]:
csv_purchase_bytes = orders_csv.memory_usage(deep=True)["order_purchase_timestamp"]
db_purchase_bytes = orders.memory_usage(deep=True)["order_purchase_timestamp"]

round(csv_purchase_bytes / db_purchase_bytes, 1)

np.float64(8.5)

The purchase timestamp column uses 6,761,988 bytes in the CSV-loaded
DataFrame and 795,528 bytes in the PostgreSQL-loaded DataFrame.

The text representation uses 8.5 times as much memory as the datetime
representation in this measurement.

## Part 1 — Types That Fit the Data

Explore categorical values, nullable integers, and explicit datetime
conversion to represent the data appropriately.

In [10]:
orders["order_status"].nunique(), len(orders)

(8, 99441)

The order_status column has only 8 distinct values across 99,441 orders,
making it a suitable candidate for categorical storage.

In [11]:
status = orders["order_status"]

status.memory_usage(deep=True), status.astype("category").memory_usage(deep=True)

(5766064, 100033)

I compared the memory usage of order_status before and after categorical
conversion. Category stores the distinct labels with codes for the rows,
which can reduce memory usage for repeated values.

In [12]:
products = query("SELECT * FROM products")

products[["product_id", "product_weight_g", "product_photos_qty"]].head(3)

,product_id,product_weight_g,product_photos_qty
0,1e9e8ef04dbcff4541ed26657ea517e5,225.0,1.0
1,3aa071139cb16b67ca9e5dea641aaa2f,1000.0,1.0
2,96bd76ec8810374ed1b65e291975717f,154.0,1.0


In [13]:
products["product_weight_g"].isna().sum(), products["product_photos_qty"].isna().sum()

(np.int64(2), np.int64(610))

In [14]:
products = products.astype({
    "product_weight_g": "Int64",
    "product_photos_qty": "Int64",
})

products[["product_id", "product_weight_g", "product_photos_qty"]].head(3)

,product_id,product_weight_g,product_photos_qty
0,1e9e8ef04dbcff4541ed26657ea517e5,225,1
1,3aa071139cb16b67ca9e5dea641aaa2f,1000,1
2,96bd76ec8810374ed1b65e291975717f,154,1


I converted product_weight_g and product_photos_qty to the nullable Int64
type. This represents whole numbers while preserving missing values as
<NA>, without replacing them with zero.

In [15]:
#pd.Series([12.5, 3.0]).astype("Int64")

The intentional conversion raised a TypeError because 12.5 cannot be
represented as an integer without changing its value.

The failing line was commented out after observing the error so that
Restart and Run All can complete.

In [16]:
text = pd.Series(["03/04/2018"])

guessed = pd.to_datetime(text)
told = pd.to_datetime(text, format="%d/%m/%Y")

guessed[0], told[0]

(Timestamp('2018-03-04 00:00:00'), Timestamp('2018-04-03 00:00:00'))

The date text 03/04/2018 is ambiguous. Specifying format="%d/%m/%Y"
explicitly interprets it as 3 April 2018 instead of relying on inference.

In [17]:
purchase = pd.to_datetime(
    orders_csv["order_purchase_timestamp"],
    format="%Y-%m-%d %H:%M:%S",
)

purchase.head(3)

0   2017-10-02 10:56:33
1   2018-07-24 20:41:37
2   2018-08-08 08:38:49
Name: order_purchase_timestamp, dtype: datetime64[us]

I converted the purchase timestamp text using the explicit format
%Y-%m-%d %H:%M:%S. The converted Series is stored in purchase;
orders_csv remains unchanged.

In [18]:
#a.1
fixed = orders_csv.copy()

for column in orders.columns[3:]:
    fixed[column] = pd.to_datetime(
        fixed[column], format="%Y-%m-%d %H:%M:%S"
    )

fixed = fixed.astype({"order_status": "category"})
megabytes(fixed), fixed.equals(smaller)

(np.float64(20.2), True)

### Exercise 1.a — Explicit Types

I copied orders_csv, converted its five timestamp columns to datetimes,
and converted order_status to category.

The original orders_csv was not modified. I compared the memory usage
and used equals to check whether fixed matches smaller.

In [19]:
#b.1
customers = query("SELECT * FROM customers")
print("Before:", megabytes(customers), "MB")


customers = customers.astype({
    "customer_state": "category",
    "customer_city": "category",
})
print("After:", megabytes(customers), "MB")


customer_id = customers["customer_id"]
print("ID before:", customer_id.memory_usage(deep=True))
print("ID after:", customer_id.astype("category").memory_usage(deep=True))

Before: 32.5 MB
After: 22.0 MB
ID before: 8054853
ID after: 8452617


### Exercise 1.b — Choosing Suitable Categorical Columns

I measured customer data before and after converting customer_state and
customer_city to category. I also measured customer_id separately.

State and city contain repeated values. In contrast, customer_id is unique
for each customer record. Categorical storage must keep the unique IDs
plus row codes, so it does not provide the same memory benefit and can
use more memory.

In [20]:
#c.1
missing_before = products.isna().sum()


columns = products.select_dtypes(include="float64").columns
products = products.astype({column: "Int64" for column in columns})

display(products[columns].dtypes)
display(products[columns].isna().sum())
print("Missing counts unchanged:", missing_before.equals(products.isna().sum()))

product_name_lenght           Int64
product_description_lenght    Int64
product_length_cm             Int64
product_height_cm             Int64
product_width_cm              Int64
dtype: object

product_name_lenght           610
product_description_lenght    610
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Missing counts unchanged: True


### Exercise 1.c — Nullable Integer Columns

I converted product_name_lenght, product_description_lenght,
product_length_cm, product_height_cm, and product_width_cm to Int64.

I compared missing-value counts before and after conversion to verify
that changing the types did not remove or replace missing values.

## Part 2 — Merges That Check Themselves

Use validate to check merge-key uniqueness and indicator to identify
matched and unmatched rows.

In [21]:
items = query("SELECT * FROM order_items")
payments = query("SELECT * FROM order_payments")

In [22]:
# Expect one payment row per order on the right.
#items.merge(payments, on="order_id", validate="many_to_one")

The merge raised a MergeError because order_id is not unique in payments.
The many_to_one validation detected that the expected relationship was wrong.
The intentional failing line was commented out after observing the error.


In [23]:
# many_to_many allows the merge without uniqueness checks.
len(items), len(items.merge(
    payments, on="order_id", validate="many_to_many"
))

(112650, 117601)

Allowing the items-to-payments merge without uniqueness checks can duplicate
item rows when an order has multiple payment records.
The many_to_many option does not validate key uniqueness.

In [24]:
# Each item should match one order.
len(items), len(items.merge(
    orders, on="order_id", how="left", validate="many_to_one"
))

(112650, 112650)

The items-to-orders merge uses many_to_one validation because multiple
items can belong to one order. The left merge preserves unmatched items,
while validation prevents duplication caused by repeated order keys.

In [25]:
translation = query("SELECT * FROM category_translation")

products_en = products.merge(translation,on="product_category_name",how="left",validate="many_to_one",indicator=True)

products_en["_merge"].value_counts()

_merge
both          32328
left_only       623
right_only        0
Name: count, dtype: int64

In [26]:
products_en.loc[products_en["_merge"] == "left_only","product_category_name",].value_counts(dropna=False)

product_category_name
NaN                                              610
portateis_cozinha_e_preparadores_de_alimentos     10
pc_gamer                                           3
Name: count, dtype: int64

The merge indicator identifies products without a matching category
translation. Using value_counts(dropna=False) includes missing category
names, helping distinguish missing categories from untranslated categories.

In [27]:
left = pd.DataFrame({
    "category": ["toys", None],
    "items": [5, 3],
})

right = pd.DataFrame({
    "category": ["toys", None],
    "label": ["Toys", "No category"],
})

left.merge( right, on="category", how="left", validate="one_to_one")

,category,items,label
0,toys,5,Toys
1,NaN,3,No category


Pandas matches missing merge keys with each other.
In SQL, a standard equality join does not match NULL keys.
Missing keys should therefore be checked before interpreting a merge.

In [28]:
#a.2
payments_orders = payments.merge(
    orders, on="order_id", how="inner", validate="many_to_one")

len(payments), len(payments_orders)

(103886, 103886)

### Exercise 2.a — Payments to Orders

The merge uses many_to_one validation because an order can have multiple
payment records, while order_id is unique in orders.

The row count stayed unchanged: 103,886 before and 103,886 after the
inner merge. Every payment record found a matching order.

In [29]:
#b.2
orders_payments = orders.merge(
    payments,
    on="order_id",
    how="left",
    validate="one_to_many",
    indicator=True,
)

# Select orders with no matching payment.
no_payment = orders_payments[orders_payments["_merge"] == "left_only"]

print("Orders without payment:", len(no_payment))
no_payment[["order_id", "order_status"]]

Orders without payment: 1


,order_id,order_status
32133,bfbd0f9bdef84302105ad712db648a6c,delivered


### Exercise 2.b — Orders Without Payment Records

I used a left merge with one_to_many validation and indicator=True.
Filtering for left_only identified one order without a matching payment record:

- Order ID: bfbd0f9bdef84302105ad712db648a6c
- Order status: delivered

This shows a missing payment record in the dataset; it does not prove
that the customer did not pay.

In [30]:
#c.2
left.merge(
    right, on="category", how="inner", validate="one_to_one")

,category,items,label
0,toys,5,Toys
1,NaN,3,No category


### Exercise 2.c — Missing Keys in an Inner Merge

The inner merge returned two rows: toys and the missing category.
Pandas matched the missing keys, giving that row the label "No category".

A SQL INNER JOIN using ordinary equality would return only the toys row,
because NULL keys do not match through an equality condition.

## Part 3 — Aggregation and Transformation

Use agg to produce one summary row per order, and transform to return
a group-level value for every original item row.

In [31]:
# Summarize item counts and prices for each order.
per_order = items.groupby("order_id").agg(
    items=("order_item_id", "size"),
    items_total=("price", "sum"),
)

len(items), len(per_order)

(112650, 98666)

In [52]:
per_order[per_order["items"] >= 3].head(5)

,items,items_total
order_id,,
00143d0f86d6fbd9f9b38ab440ac16f5,3,63.99
001ab0a7578dd66cd4b0a71f5b6e1e41,3,74.67
00526a9d4ebde463baee25f386963ddc,4,135.56
005d9a5423d47281ac463a968b3936fb,3,99.97
00a57dfbb049fbaae10763e2cf15f797,3,269.70


Named aggregation produces one row per order that has items.
The items column counts item rows, and items_total sums their prices.
Orders without item records are not included in this summary.

In [33]:
# Add each order's total to all its item rows.
items["order_total"] = items.groupby("order_id")["price"].transform("sum")

# Calculate each item's percentage of its order total.
items["pct_of_order"] = (100 * items["price"] / items["order_total"]).round(1)

# Inspect the same example order used on Day 3.
items[
    items["order_id"] == "06d5cbe68295558fd6c48998db3ea397"
][["order_item_id", "price", "order_total", "pct_of_order"]]

,order_item_id,price,order_total,pct_of_order
3037,1,39.0,123.0,31.7
3038,2,33.0,123.0,26.8
3039,3,51.0,123.0,41.5


Transform keeps every item row and repeats its order total alongside it.
For the example order, the three item prices total 123.0.
The item priced at 51.0 represents 41.5% of that total.

This is equivalent to SUM(price) OVER (PARTITION BY order_id) in SQL.

In [34]:
# Keep items whose price equals the maximum price in their order.
top_items = items[
    items["price"] == items.groupby("order_id")["price"].transform("max")
]

len(top_items), top_items["order_id"].nunique()


(109491, 98666)

I selected item rows whose price equals the maximum price within their
order. I compared the number of selected rows with the number of distinct
orders to check whether the result contains one row per order.

In [35]:
#a.3
items["items_in_order"] = items.groupby("order_id")["order_item_id"].transform("size")

print("Items in multi-item orders:", len(items[items["items_in_order"] >= 2]))
print("Multi-item orders:", len(per_order[per_order["items"] >= 2]))

Items in multi-item orders: 23787
Multi-item orders: 9803


### Exercise a.3 — Items and Orders

I used transform("size") to add each order's item count to every item row.

The first result counts item rows belonging to orders with at least two
items. The second result counts the orders themselves, using per_order.

In [36]:
#b.3
print("Repeated order totals:", round(items["order_total"].sum(), 2))
print("Item prices:", round(items["price"].sum(), 2))
print("One total per order:", round(per_order["items_total"].sum(), 2))

Repeated order totals: 17230898.37
Item prices: 13591643.7
One total per order: 13591643.7


### Exercise b.3 — Comparing Totals

Summing item prices and summing one total per order give the same overall
amount, apart from possible floating-point rounding differences.

Summing order_total across item rows overcounts the amount because each
order's total is repeated once for every item in that order.

In [37]:
top_counts = top_items["order_id"].value_counts()
example_order = top_counts[top_counts > 1].index[0]

print("Order ID:", example_order)
print("Top-priced items:", top_counts.loc[example_order])

items.loc[
    items["order_id"] == example_order,["order_id", "order_item_id", "price"],].sort_values("order_item_id")

Order ID: 1b15974a0141d54e36626dca3fdc731a
Top-priced items: 20


,order_id,order_item_id,price
11932,1b15974a0141d54e36626dca3fdc731a,1,100.0
11933,1b15974a0141d54e36626dca3fdc731a,2,100.0
11934,1b15974a0141d54e36626dca3fdc731a,3,100.0
11935,1b15974a0141d54e36626dca3fdc731a,4,100.0
11936,1b15974a0141d54e36626dca3fdc731a,5,100.0
11937,1b15974a0141d54e36626dca3fdc731a,6,100.0
11938,1b15974a0141d54e36626dca3fdc731a,7,100.0
11939,1b15974a0141d54e36626dca3fdc731a,8,100.0
11940,1b15974a0141d54e36626dca3fdc731a,9,100.0
11941,1b15974a0141d54e36626dca3fdc731a,10,100.0


### Exercise c.3 — Tied Maximum Prices

I selected an order with multiple rows in top_items and displayed all
its items to inspect their prices.

Multiple items are kept when they share the highest price in the order.
This behaves like RANK() = 1 when ranking by price in descending order.
ROW_NUMBER() = 1 would select only one row.

## Part 4 — Long Tables and Wide Tables

Use melt to turn event columns into rows, then use pivot_table to
summarize event counts by month.

In [38]:
times = query("""
    SELECT order_id,
           order_purchase_timestamp AS purchased,
           order_approved_at AS approved,
           order_delivered_carrier_date AS to_carrier,
           order_delivered_customer_date AS delivered
    FROM orders
""")

times.head(3)

,order_id,purchased,approved,to_carrier,delivered
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29


The wide table contains one row per order, with separate timestamp
columns for purchase, approval, carrier handover, and customer delivery.

In [39]:
events = times.melt(
    id_vars="order_id",
    var_name="event",
    value_name="time",
)

len(times), len(events)

(99441, 397764)

melt converts the four event columns into event and time columns.
Each order now has four rows, including events with missing timestamps.
The row count increases from 99,441 to 397,764.

In [40]:
events[
    events["order_id"] == "06d5cbe68295558fd6c48998db3ea397"
]

,order_id,event,time
35954,06d5cbe68295558fd6c48998db3ea397,purchased,2018-03-27 14:04:31
135395,06d5cbe68295558fd6c48998db3ea397,approved,2018-03-27 14:15:12
234836,06d5cbe68295558fd6c48998db3ea397,to_carrier,2018-03-28 21:25:17
334277,06d5cbe68295558fd6c48998db3ea397,delivered,2018-03-29 19:24:35


The selected order appears once for each event: purchased, approved,
to_carrier, and delivered. Each row stores the corresponding timestamp.

In [41]:
events["month"] = events["time"].dt.strftime("%Y-%m")

The month column is derived from each event's own timestamp.
A purchase and its delivery can therefore belong to different months.

In [42]:
oct_to_dec = events[
    events["month"].isin(["2017-10", "2017-11", "2017-12"])
]

counts = oct_to_dec.pivot_table(
    index="month",
    columns="event",
    values="order_id",
    aggfunc="count",
)

counts[["purchased", "approved", "to_carrier", "delivered"]]

event,purchased,approved,to_carrier,delivered
month,,,,
2017-10,4631,4590,4482,4494
2017-11,7544,7395,6637,4670
2017-12,5673,5832,6081,7205


The pivot table counts events by month and event type for October through
December 2017. Columns are displayed in the order of the order lifecycle.

These are monthly event counts, not delivery outcomes for the same
purchase-month group. Some deliveries may belong to earlier purchases.

In [43]:
#a.4
print("Missing event timestamps:", events["time"].isna().sum())

display(events["time"].isna().groupby(events["event"]).sum())

times.isna().sum()

Missing event timestamps: 4908


event
approved       160
delivered     2965
purchased        0
to_carrier    1783
Name: time, dtype: int64

order_id         0
purchased        0
approved       160
to_carrier    1783
delivered     2965
dtype: int64

### Exercise a.4 — Missing Timestamps

I counted missing timestamps overall and by event, then compared them
with the original timestamp columns in times. The counts should match
because melt preserves missing values.

In [44]:
#b.4
payments_monthly = payments.merge(
    orders[["order_id", "order_purchase_timestamp"]],
    on="order_id",
    validate="many_to_one",
)

payments_monthly["month"] = (
    payments_monthly["order_purchase_timestamp"].dt.strftime("%Y-%m")
)

payments_monthly[
    payments_monthly["month"].isin(["2017-10", "2017-11", "2017-12"])
].pivot_table(
    index="month",
    columns="payment_type",
    values="payment_value",
    aggfunc="sum",
    fill_value=0,
).round(2)

payment_type,boleto,credit_card,debit_card,voucher
month,,,,
2017-10,140633.45,614327.89,5776.57,18939.97
2017-11,226292.34,942856.67,6168.10,19565.69
2017-12,165584.05,682859.59,7125.34,22832.50


### Exercise 4.b — Payments by Purchase Month

I added each order's purchase month to its payment records using a
validated merge. The pivot table sums payments by purchase month and
payment type for October–December 2017, rounded to two decimal places.


## Part 5 — Method Chaining and Category Growth

Compare item counts between October and November 2017 using one chain.
Include categories with at least 100 items in October.

In [45]:
months = ["2017-10", "2017-11"]
min_items = 100

The comparison months and minimum October item count are defined in one
place. The minimum count reduces the influence of very small categories.

In [46]:
growth = (
    items
    .merge(orders, on="order_id", how="left", validate="many_to_one")
    .merge(products, on="product_id", how="left", validate="many_to_one")
    .merge(
        translation,
        on="product_category_name",
        how="left",
        validate="many_to_one",
    )
    .assign(
        category=lambda d: d["product_category_name_english"].fillna("unknown"),
        month=lambda d: d["order_purchase_timestamp"].dt.strftime("%Y-%m"),
    )
    .query("month in @months")
    .pivot_table(
        index="category",
        columns="month",
        values="order_item_id",
        aggfunc="count",
        fill_value=0,
    )
    .rename(columns={months[0]: "before", months[1]: "after"})
    .assign(growth=lambda d: (d["after"] / d["before"]).round(2))
    .query("before >= @min_items")
    .sort_values(["growth", "category"], ascending=[False, True])
)

growth

month,before,after,growth
category,,,
furniture_decor,379,783,2.07
garden_tools,281,550,1.96
fashion_bags_accessories,110,200,1.82
housewares,235,423,1.80
bed_bath_table,545,975,1.79
auto,150,263,1.75
toys,290,495,1.71
computers_accessories,322,532,1.65
perfumery,189,311,1.65


The chain joins item records to orders, products, and category translations
using validated merges. It counts items by category and purchase month,
then compares November with October.

Growth is the November count divided by the October count, not a percentage
increase. All order statuses are included. Categories must have at least
100 items in October, and ties in rounded growth are sorted by category name.

In [47]:
#a.5
monthly_items = (
    items
    .merge(
        orders[["order_id", "order_purchase_timestamp"]],
        on="order_id",
        validate="many_to_one",
    )
    .assign(month=lambda d: d["order_purchase_timestamp"].dt.strftime("%Y-%m"))
    .groupby("month")
    .size()
)

overall_growth = monthly_items[months[1]] / monthly_items[months[0]]

faster_categories = (
    growth["after"] / growth["before"] > overall_growth
).sum()

display(monthly_items.loc[months])
print("Overall growth:", round(overall_growth, 2))
print("Categories growing faster:", faster_categories)

month
2017-10    5322
2017-11    8665
dtype: int64

Overall growth: 1.63
Categories growing faster: 10


### Exercise 5.a — Overall Item Growth

I counted all item rows by purchase month and divided November's count
by October's count. All order statuses were included.

I then counted categories in growth whose growth ratio exceeded the
overall ratio, using unrounded ratios for the comparison.

## Part 6 — A Short Look at Polars

Rebuild the category-growth table in Polars, compare it with pandas,
and measure CSV reading time with both libraries.

In [48]:
import polars as pl

pl.Config.set_tbl_rows(30)

items_pl = pl.read_csv(RAW_OLIST / "olist_order_items_dataset.csv")
orders_pl = pl.read_csv(
    RAW_OLIST / "olist_orders_dataset.csv",
    try_parse_dates=True,
)
products_pl = pl.read_csv(RAW_OLIST / "olist_products_dataset.csv")
translation_pl = pl.read_csv(
    RAW_OLIST / "product_category_name_translation.csv"
)

orders_pl.schema

Schema([('order_id', String),
        ('customer_id', String),
        ('order_status', String),
        ('order_purchase_timestamp', Datetime(time_unit='us', time_zone=None)),
        ('order_approved_at', Datetime(time_unit='us', time_zone=None)),
        ('order_delivered_carrier_date',
         Datetime(time_unit='us', time_zone=None)),
        ('order_delivered_customer_date',
         Datetime(time_unit='us', time_zone=None)),
        ('order_estimated_delivery_date',
         Datetime(time_unit='us', time_zone=None))])

I loaded the raw CSV files directly with Polars and inspected the orders
schema to check the inferred data types. Date parsing was enabled for
the order timestamps.

In [49]:
growth_pl = (
    items_pl
    .join(orders_pl, on="order_id", how="left", validate="m:1")
    .join(products_pl, on="product_id", how="left", validate="m:1")
    .join(
        translation_pl,
        on="product_category_name",
        how="left",
        validate="m:1",
    )
    .with_columns(
        category=pl.col("product_category_name_english").fill_null("unknown"),
        month=pl.col("order_purchase_timestamp").dt.strftime("%Y-%m"),
    )
    .filter(pl.col("month").is_in(months))
    .pivot(
        on="month",
        index="category",
        values="order_item_id",
        aggregate_function="len",
    )
    .fill_null(0)
    .rename({months[0]: "before", months[1]: "after"})
    .with_columns(
        growth=(pl.col("after") / pl.col("before")).round(2)
    )
    .filter(pl.col("before") >= min_items)
    .select("category", "before", "after", "growth")
    .sort(["growth", "category"], descending=[True, False])
)

growth_pl

category,before,after,growth
str,u32,u32,f64
"""furniture_decor""",379,783,2.07
"""garden_tools""",281,550,1.96
"""fashion_bags_accessories""",110,200,1.82
"""housewares""",235,423,1.8
"""bed_bath_table""",545,975,1.79
"""auto""",150,263,1.75
"""toys""",290,495,1.71
"""computers_accessories""",322,532,1.65
"""perfumery""",189,311,1.65


I rebuilt the Part 5 analysis in Polars using the same months, minimum
October count, and sorting rules. All joins use m:1 validation.

The table counts items across all order statuses and calculates the
November-to-October growth ratio.


### Exercise 6.a — Incorrect Join Validation

I temporarily changed the first join validation from m:1 to 1:1.
The join failed because order_id repeats in the item records.

I restored m:1 after observing the error and rebuilt the result.

In [50]:
same_categories = growth_pl["category"].to_list() == growth.index.tolist()
same_growth = growth_pl["growth"].to_list() == growth["growth"].tolist()

same_categories, same_growth

(True, True)

I compared the ordered category names and growth ratios between Polars
and pandas. These checks verify agreement for those two outputs.

In [51]:
import time

start = time.perf_counter()
pd.read_csv(RAW_OLIST / "olist_geolocation_dataset.csv")
pandas_seconds = time.perf_counter() - start

start = time.perf_counter()
pl.read_csv(RAW_OLIST / "olist_geolocation_dataset.csv")
polars_seconds = time.perf_counter() - start

print("Pandas seconds:", round(pandas_seconds, 2))
print("Polars seconds:", round(polars_seconds, 2))
print("Pandas / Polars time ratio:", round(pandas_seconds / polars_seconds, 1))

Pandas seconds: 0.68
Polars seconds: 0.09
Pandas / Polars time ratio: 7.9


I measured CSV reading time for the geolocation file with both libraries
and repeated the measurement three times.

The first run may differ because of file caching. These measurements
describe CSV reading on this machine, not the speed of every operation.